# Hyperparameter Optimization Notebook

This notebook performs systematic hyperparameter tuning on the UCI-HAR task using the same improved preprocessing logic, then evaluates tuned models on the held-out test set.

## Workflow
1. Load train/test data.
2. Apply improved preprocessing (variance filter + SelectKBest + correlation pruning).
3. Tune candidate models with cross-validated randomized/grid search.
4. Evaluate tuned models on test data.
5. Save tuning and final results to CSV.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_score

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

try:
    from xgboost import XGBClassifier
except Exception:
    XGBClassifier = None

RANDOM_STATE = 42

In [2]:
# Load prepared files from your prior workflow
X_train = pd.read_csv("X_train.csv")
X_test = pd.read_csv("X_test.csv")
y_train = pd.read_csv("y_train.csv").values.ravel()
y_test = pd.read_csv("y_test.csv").values.ravel()

# Keep labels compatible with models that require 0..K-1 labels (for example, XGBoost)
if np.min(y_train) == 1 and np.max(y_train) == 6:
    y_train = y_train - 1
    y_test = y_test - 1

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Classes:", np.unique(y_train))

Train shape: (7352, 562)
Test shape: (2947, 562)
Classes: [0 1 2 3 4 5]


In [3]:
# Improved preprocessing: variance filtering + SelectKBest + correlation pruning
VARIANCE_THRESHOLD = 0.01

var_selector = VarianceThreshold(threshold=VARIANCE_THRESHOLD)
X_train_clean = var_selector.fit_transform(X_train)
X_test_clean = var_selector.transform(X_test)

print(f"After variance filtering: {X_train_clean.shape[1]} features")

best_k = 525
print("Best k from sweep:", best_k)

kb_selector = SelectKBest(score_func=f_classif, k=best_k)
X_train_kb = kb_selector.fit_transform(X_train_clean, y_train)
X_test_kb = kb_selector.transform(X_test_clean)

# Correlation pruning
corr_matrix = np.corrcoef(X_train_kb, rowvar=False)
cols_to_drop = set()
for i in range(corr_matrix.shape[0]):
    if i in cols_to_drop:
        continue
    for j in range(i + 1, corr_matrix.shape[1]):
        if abs(corr_matrix[i, j]) > 0.95:
            cols_to_drop.add(j)

keep_mask = np.array([idx not in cols_to_drop for idx in range(X_train_kb.shape[1])])
X_train_sel = X_train_kb[:, keep_mask]
X_test_sel = X_test_kb[:, keep_mask]

print(f"After SelectKBest: {X_train_kb.shape[1]} features")
print(f"After correlation pruning: {X_train_sel.shape[1]} features")
print("Final shapes:", X_train_sel.shape, X_test_sel.shape)

After variance filtering: 525 features
Best k from sweep: 525
After SelectKBest: 525 features
After correlation pruning: 257 features
Final shapes: (7352, 257) (2947, 257)


In [10]:
from sklearn.svm import LinearSVC

# Candidate models with compact parameter candidates for faster tuning
quick_candidates = {
    "Logistic Regression": (
        LogisticRegression(max_iter=3000, random_state=RANDOM_STATE, n_jobs=-1),
        [
            {"C": 0.3, "solver": "lbfgs"},
            {"C": 1.0, "solver": "lbfgs"},
            {"C": 3.0, "solver": "saga"},
        ],
    ),
    "Linear SVC": (
        LinearSVC(random_state=RANDOM_STATE, max_iter=1000),
        [
            {"C": 0.3},
            {"C": 1.0},
            {"C": 3.0},
        ],
    ),
    "Random Forest": (
        RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
        [
            {"n_estimators": 200, "max_depth": None, "min_samples_split": 2, "min_samples_leaf": 1, "max_features": "sqrt"},
            {"n_estimators": 300, "max_depth": 20, "min_samples_split": 4, "min_samples_leaf": 2, "max_features": "sqrt"},
        ],
    ),
    "Hist Gradient Boosting": (
        HistGradientBoostingClassifier(random_state=RANDOM_STATE),
        [
            {"learning_rate": 0.05, "max_depth": 8, "max_iter": 200, "min_samples_leaf": 20, "l2_regularization": 0.001},
            {"learning_rate": 0.08, "max_depth": 10, "max_iter": 300, "min_samples_leaf": 30, "l2_regularization": 0.01},
        ],
    ),
}

if XGBClassifier is not None:
    quick_candidates["XGBoost"] = (
        XGBClassifier(
            objective="multi:softprob",
            eval_metric="mlogloss",
            num_class=len(np.unique(y_train)),
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        [
            {"n_estimators": 200, "max_depth": 5, "learning_rate": 0.08, "subsample": 0.8, "colsample_bytree": 0.8, "reg_lambda": 1.0},
            {"n_estimators": 300, "max_depth": 6, "learning_rate": 0.05, "subsample": 0.9, "colsample_bytree": 0.9, "reg_lambda": 2.0},
        ],
    )

print("Models to tune quickly:", list(quick_candidates.keys()))

Models to tune quickly: ['Logistic Regression', 'Linear SVC', 'Random Forest', 'Hist Gradient Boosting', 'XGBoost']


In [11]:
# Fast hyperparameter tuning with compact candidate sets + 3-fold stratified CV
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

search_results = []
best_estimators = {}

for name, (base_model, candidate_params) in quick_candidates.items():
    print("=" * 70)
    print(f"Tuning : {name}")

    best_score = -np.inf
    best_params = None
    best_model = None

    for params in candidate_params:
        model = clone(base_model).set_params(**params)
        scores = cross_val_score(
            model,
            X_train_sel,
            y_train,
            scoring="accuracy",
            cv=cv,
            n_jobs=-1,
        )
        mean_score = scores.mean()

        if mean_score > best_score:
            best_score = mean_score
            best_params = params
            best_model = clone(base_model).set_params(**params)

    best_model.fit(X_train_sel, y_train)
    best_estimators[name] = best_model
    search_results.append(
        {
            "Model": name,
            "Best CV Accuracy": best_score,
            "Best Params": best_params,
        }
    )

    print(f"Best CV accuracy: {best_score:.4f}")
    print(f"Best params: {best_params}")

tuning_df = pd.DataFrame(search_results).sort_values("Best CV Accuracy", ascending=False).reset_index(drop=True)
tuning_df

Tuning : Logistic Regression


c:\Users\User\Desktop\Kaggle\HAR\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Best CV accuracy: 0.9773
Best params: {'C': 3.0, 'solver': 'saga'}
Tuning : Linear SVC
Best CV accuracy: 0.9781
Best params: {'C': 0.3}
Tuning : Random Forest
Best CV accuracy: 0.9793
Best params: {'n_estimators': 200, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt'}
Tuning : Hist Gradient Boosting
Best CV accuracy: 0.9946
Best params: {'learning_rate': 0.08, 'max_depth': 10, 'max_iter': 300, 'min_samples_leaf': 30, 'l2_regularization': 0.01}
Tuning : XGBoost
Best CV accuracy: 0.9916
Best params: {'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.08, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_lambda': 1.0}


,Model,Best CV Accuracy,Best Params
0,Hist Gradient Boosting,0.994559,"{'learning_rate': 0.08, 'max_depth': 10, 'max_..."
1,XGBoost,0.991567,"{'n_estimators': 200, 'max_depth': 5, 'learnin..."
2,Random Forest,0.979325,"{'n_estimators': 200, 'max_depth': None, 'min_..."
3,Linear SVC,0.978101,{'C': 0.3}
4,Logistic Regression,0.977285,"{'C': 3.0, 'solver': 'saga'}"


In [12]:
# Final held-out test evaluation of tuned models
final_rows = []

for name, model in best_estimators.items():
    pred = model.predict(X_test_sel)
    final_rows.append(
        {
            "Model": name,
            "Accuracy": accuracy_score(y_test, pred),
            "Precision": precision_score(y_test, pred, average="weighted"),
            "Recall": recall_score(y_test, pred, average="weighted"),
            "F1-Score": f1_score(y_test, pred, average="weighted"),
        }
    )

final_df = pd.DataFrame(final_rows).sort_values("Accuracy", ascending=False).reset_index(drop=True)
final_df

,Model,Accuracy,Precision,Recall,F1-Score
0,Linear SVC,0.952494,0.954573,0.952494,0.952687
1,Logistic Regression,0.950119,0.951926,0.950119,0.950092
2,XGBoost,0.942314,0.943370,0.942314,0.942147
3,Random Forest,0.940957,0.941953,0.940957,0.940717
4,Hist Gradient Boosting,0.936885,0.937598,0.936885,0.936766


In [13]:
# Save outputs for reporting
tuning_export = tuning_df.copy()
tuning_export["Best Params"] = tuning_export["Best Params"].astype(str)

tuning_export.to_csv("hyperparameter_tuning_cv.csv", index=False)
final_df.to_csv("hyperparameter_tuned_test.csv", index=False)

print("Saved: hyperparameter_tuning_cv.csv")
print("Saved: hyperparameter_tuned_test.csv")

print("\nTop tuned model on held-out test set:")
print(final_df.iloc[0].to_string())

Saved: hyperparameter_tuning_cv.csv
Saved: hyperparameter_tuned_test.csv

Top tuned model on held-out test set:
Model        Linear SVC
Accuracy       0.952494
Precision      0.954573
Recall         0.952494
F1-Score       0.952687


In [14]:
best_estimators

{'Logistic Regression': LogisticRegression(C=3.0, max_iter=3000, n_jobs=-1, random_state=42,
                    solver='saga'),
 'Linear SVC': LinearSVC(C=0.3, random_state=42),
 'Random Forest': RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42),
 'Hist Gradient Boosting': HistGradientBoostingClassifier(l2_regularization=0.01, learning_rate=0.08,
                                max_depth=10, max_iter=300, min_samples_leaf=30,
                                random_state=42),
 'XGBoost': XGBClassifier(base_score=None, booster=None, callbacks=None,
               colsample_bylevel=None, colsample_bynode=None,
               colsample_bytree=0.8, device=None, early_stopping_rounds=None,
               enable_categorical=False, eval_metric='mlogloss',
               feature_types=None, feature_weights=None, gamma=None,
               grow_policy=None, importance_type=None,
               interaction_constraints=None, learning_rate=0.08, max_bin=None,
               max_

In [18]:
best_estimators.keys()

dict_keys(['Logistic Regression', 'Linear SVC', 'Random Forest', 'Hist Gradient Boosting', 'XGBoost'])

In [ ]:
from models import get_models, get_improved_models

models = get_models()
models.update(get_improved_models())
tuned_models = models.copy()

for name in best_estimators.keys():
    tuned_models[name] = best_estimators[name]



{'Decision Tree': DecisionTreeClassifier(random_state=42), 'Random Forest': RandomForestClassifier(n_jobs=-1, random_state=42), 'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42), 'Linear SVC': LinearSVC(), 'RBF SVM': SVC(C=10, random_state=42), 'K-Nearest Neighbor': KNeighborsClassifier(), 'Extra Trees': ExtraTreesClassifier(n_estimators=300, n_jobs=-1, random_state=42), 'Hist Gradient Boosting': HistGradientBoostingClassifier(max_iter=300, random_state=42), 'Tuned Random Forest': RandomForestClassifier(n_estimators=500, n_jobs=-1, random_state=42), 'Tuned Logistic Regression': LogisticRegression(C=10, max_iter=2000, n_jobs=-1, random_state=42,
                   solver='saga'), 'Tuned RBF SVM': SVC(C=100, random_state=42), 'XGBoost': XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.9, device=None, early_stopping_rounds=None,
              enable_categorical

## Robust Validation of Tuned Models

This section evaluates tuned models using two complementary validation techniques:
1. **Stratified 5-fold cross-validation** on training data (mean and std).
2. **Held-out test set evaluation** for final generalization performance.

Reported metrics: Accuracy, Precision (weighted), Recall (weighted), F1-score (weighted), and Balanced Accuracy.

In [26]:
from sklearn.model_selection import cross_validate
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, classification_report

# Evaluate only tuned models selected in the hyperparameter search
tuned_model_names = list(best_estimators.keys())
cv_eval = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

scoring = {
    "accuracy": "accuracy",
    "precision_weighted": "precision_weighted",
    "recall_weighted": "recall_weighted",
    "f1_weighted": "f1_weighted",
    "balanced_accuracy": "balanced_accuracy",
}

cv_rows = []
test_rows = []

for model_name in tuned_model_names:
    model = best_estimators[model_name]

    cv_scores = cross_validate(
        model,
        X_train_sel,
        y_train,
        cv=cv_eval,
        scoring=scoring,
        n_jobs=-1,
    )

    cv_rows.append(
        {
            "Model": model_name,
            "CV Accuracy Mean": cv_scores["test_accuracy"].mean(),
            "CV Accuracy Std": cv_scores["test_accuracy"].std(),
            "CV Precision Mean": cv_scores["test_precision_weighted"].mean(),
            "CV Recall Mean": cv_scores["test_recall_weighted"].mean(),
            "CV F1 Mean": cv_scores["test_f1_weighted"].mean(),
            "CV Balanced Acc Mean": cv_scores["test_balanced_accuracy"].mean(),
        }
    )

    y_pred = model.predict(X_test_sel)
    test_rows.append(
        {
            "Model": model_name,
            "Test Accuracy": accuracy_score(y_test, y_pred),
            "Test Precision": precision_score(y_test, y_pred, average="weighted"),
            "Test Recall": recall_score(y_test, y_pred, average="weighted"),
            "Test F1": f1_score(y_test, y_pred, average="weighted"),
            "Test Balanced Acc": balanced_accuracy_score(y_test, y_pred),
        }
    )

cv_eval_df = pd.DataFrame(cv_rows).sort_values("CV Accuracy Mean", ascending=False).reset_index(drop=True)
test_eval_df = pd.DataFrame(test_rows).sort_values("Test Accuracy", ascending=False).reset_index(drop=True)

cv_eval_df.to_csv("hyperparameter_tuned_cv_validation.csv", index=False)
test_eval_df.to_csv("hyperparameter_tuned_test_validation.csv", index=False)

print("Cross-validation summary (5-fold stratified):")
display(cv_eval_df)

print("Held-out test summary:")
display(test_eval_df)

print("Saved: hyperparameter_tuned_cv_validation.csv")
print("Saved: hyperparameter_tuned_test_validation.csv")

# Detailed diagnostics for best tuned model on held-out test set
best_tuned_name = test_eval_df.iloc[0]["Model"]
best_tuned_model = best_estimators[best_tuned_name]
best_pred = best_tuned_model.predict(X_test_sel)

print(f"Best tuned model on test set: {best_tuned_name}")
print("\nClassification report:")
print(classification_report(y_test, best_pred, digits=4))

cm = confusion_matrix(y_test, best_pred)
cm_df = pd.DataFrame(cm, index=[f"True_{c}" for c in np.unique(y_test)], columns=[f"Pred_{c}" for c in np.unique(y_test)])
print("Confusion matrix:")
display(cm_df)

Cross-validation summary (5-fold stratified):


,Model,CV Accuracy Mean,CV Accuracy Std,CV Precision Mean,CV Recall Mean,CV F1 Mean,CV Balanced Acc Mean
0,Hist Gradient Boosting,0.995375,0.001743,0.995388,0.995375,0.995376,0.995608
1,XGBoost,0.992519,0.001140,0.992544,0.992519,0.992518,0.992772
2,Random Forest,0.981365,0.003153,0.981453,0.981365,0.981361,0.982036
3,Linear SVC,0.978509,0.003350,0.978612,0.978509,0.978489,0.980072
4,Logistic Regression,0.978236,0.004368,0.978386,0.978236,0.978218,0.979797


Held-out test summary:


,Model,Test Accuracy,Test Precision,Test Recall,Test F1,Test Balanced Acc
0,Linear SVC,0.952494,0.954573,0.952494,0.952687,0.952987
1,Logistic Regression,0.950119,0.951926,0.950119,0.950092,0.949827
2,XGBoost,0.942314,0.943370,0.942314,0.942147,0.941024
3,Random Forest,0.940957,0.941953,0.940957,0.940717,0.938010
4,Hist Gradient Boosting,0.936885,0.937598,0.936885,0.936766,0.935979


Saved: hyperparameter_tuned_cv_validation.csv
Saved: hyperparameter_tuned_test_validation.csv
Best tuned model on test set: Linear SVC

Classification report:
              precision    recall  f1-score   support

           0     0.9591    0.9940    0.9762       496
           1     0.9805    0.9597    0.9700       471
           2     0.9976    0.9833    0.9904       420
           3     0.9421    0.8615    0.9000       491
           4     0.8591    0.9511    0.9028       532
           5     1.0000    0.9683    0.9839       537

    accuracy                         0.9525      2947
   macro avg     0.9564    0.9530    0.9539      2947
weighted avg     0.9546    0.9525    0.9527      2947

Confusion matrix:


,Pred_0,Pred_1,Pred_2,Pred_3,Pred_4,Pred_5
True_0,493,3,0,0,0,0
True_1,18,452,1,0,0,0
True_2,3,4,413,0,0,0
True_3,0,2,0,423,66,0
True_4,0,0,0,26,506,0
True_5,0,0,0,0,17,520
